# Module 3: LLM APIs & Application Development
### Complete Hands-on Guide: Days 11 – 15

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manishjhcssp-prog/ai-chat-application/blob/main/colab/module3_llm_apis.ipynb)

---

## Objectives Covered:
1. **LLM APIs & Authentication:** Environment variable security, key management, multi-provider connection (OpenAI, Gemini).
2. **Conversation & Chat History:** Managing roles (`system`, `user`, `assistant`), sliding window context truncation.
3. **Parameter Optimization:** Tuning `temperature`, `top_p`, and `max_tokens` for deterministic vs creative reasoning.
4. **Structured Responses:** Enforcing deterministic JSON outputs validated with Pydantic schemas.
5. **Robust Error Handling:** Resilient retries with exponential backoff and jitter for rate-limiting (HTTP 429).
6. **API Cost Management:** Real-time token economics, pricing calculations, and budget caps.

## 1. Install Dependencies & Setup Environment

In [ ]:
# Install core LLM libraries
!pip install -q openai google-generativeai pydantic python-dotenv tenacity

import os
import json
import time
import math
from typing import List, Dict, Any
from pydantic import BaseModel, Field

print("✅ Dependencies successfully installed.")

## 2. Secure Authentication & Environment Variables
Never hardcode API keys in plaintext code. We retrieve keys from environment variables or Colab secrets.

In [ ]:
import os
from getpass import getpass

# Attempt Colab secrets or prompt safely
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

print(f"OpenAI Key configured: {bool(OPENAI_API_KEY)}")
print(f"Gemini Key configured: {bool(GEMINI_API_KEY)}")

## 3. Conversation Management & Context Window Sliding
LLMs are stateless. To enable multi-turn memory, we send an ordered list of messages. To prevent token overflow and skyrocketing costs, we implement a **Sliding Context Window**.

In [ ]:
class ConversationManager:
    def __init__(self, system_prompt: str, max_context_tokens: int = 2000):
        self.system_prompt = {"role": "system", "content": system_prompt}
        self.history: List[Dict[str, str]] = []
        self.max_context_tokens = max_context_tokens

    @staticmethod
    def estimate_tokens(text: str) -> int:
        return max(1, math.ceil(len(text) / 4.0))

    def total_tokens(self) -> int:
        tokens = self.estimate_tokens(self.system_prompt["content"])
        for m in self.history:
            tokens += self.estimate_tokens(m["content"])
        return tokens

    def add_user_message(self, content: str):
        self.history.append({"role": "user", "content": content})
        self._prune_history()

    def add_assistant_message(self, content: str):
        self.history.append({"role": "assistant", "content": content})
        self._prune_history()

    def _prune_history(self):
        # Retain system prompt at all times; prune oldest dialogue if exceeding token budget
        while len(self.history) > 2 and self.total_tokens() > self.max_context_tokens:
            popped = self.history.pop(0)
            print(f"⚠️ Context budget exceeded! Pruned message: [{popped['role']}] {popped['content'][:30]}...")

    def get_api_payload(self) -> List[Dict[str, str]]:
        return [self.system_prompt] + self.history

# Test conversation state
conv = ConversationManager(system_prompt="You are a Senior Machine Learning Architect.", max_context_tokens=100)
conv.add_user_message("Hello, how do transformers calculate self-attention?")
conv.add_assistant_message("Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V. It computes relevance scores between all tokens.")
print(f"Current conversation turns: {len(conv.history)}")
print(f"Estimated total tokens: {conv.total_tokens()}")

## 4. Structured JSON Outputs with Pydantic Validation
Production systems require deterministic, typed data. We enforce JSON schemas and parse them with Pydantic.

In [ ]:
class IssueItem(BaseModel):
    severity: str
    description: str
    suggestion: str

class CodeAuditReport(BaseModel):
    language: str
    code_quality_score: int = Field(ge=0, le=100)
    summary: str
    issues: List[IssueItem] = Field(default_factory=list)

# Demonstration schema generation
print("Generated JSON Schema for LLM System Prompt:")
print(json.dumps(CodeAuditReport.model_json_schema(), indent=2))

# Sample simulated LLM response
sample_llm_json = '''
{
  "language": "python",
  "code_quality_score": 95,
  "summary": "Clean implementation with good abstraction.",
  "issues": [
    {
      "severity": "Low",
      "description": "Missing docstring in helper function.",
      "suggestion": "Add PEP-257 docstring explaining inputs and outputs."
    }
  ]
}
'''

validated_audit = CodeAuditReport.model_validate_json(sample_llm_json)
print(f"✅ Successfully Validated! Quality Score: {validated_audit.code_quality_score}/100")

## 5. Robust Error Handling & Exponential Backoff Retries
Handling HTTP 429 (Rate Limits) and HTTP 503 (Provider Downtime) with exponential backoff and jitter.

In [ ]:
import random
import time

def retry_api_call(func, max_retries=3, initial_delay=0.5):
    delay = initial_delay
    for attempt in range(1, max_retries + 1):
        try:
            return func()
        except Exception as ex:
            if attempt == max_retries:
                print(f"❌ Final retry failed: {ex}")
                raise
            jitter = random.uniform(0.1, 0.3)
            sleep_time = delay + jitter
            print(f"⚠️ Attempt {attempt} failed ({ex}). Retrying in {sleep_time:.2f}s...")
            time.sleep(sleep_time)
            delay *= 2

# Simulate flaky API call
attempt_count = 0
def flaky_network_call():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError("HTTP 429 Rate Limit Exceeded")
    return "✅ SUCCESS: 200 OK"

result = retry_api_call(flaky_network_call)
print(result)

## 6. Real-Time Token Economics & Cost Tracking
Auditing token expenditure per turn against predefined model pricing tiers.

In [ ]:
PRICING = {
    "gpt-4o-mini": {"prompt": 0.150, "completion": 0.600},
    "gpt-4o": {"prompt": 2.500, "completion": 10.000},
    "gemini-1.5-flash": {"prompt": 0.075, "completion": 0.300},
}

def calculate_turn_cost(model: str, prompt_toks: int, comp_toks: int) -> float:
    rates = PRICING.get(model, PRICING["gpt-4o-mini"])
    p_cost = (prompt_toks / 1_000_000) * rates["prompt"]
    c_cost = (comp_toks / 1_000_000) * rates["completion"]
    return round(p_cost + c_cost, 6)

# Benchmark cost comparison for a 1,000 prompt / 500 completion turn
for model in PRICING:
    cost = calculate_turn_cost(model, 1000, 500)
    print(f"Model: {model:<18} | Cost: ${cost:.6f} USD")

print("\n💡 Notice how gpt-4o-mini and gemini-1.5-flash are over 10x-15x more cost-efficient than flagship models!")